# CAISO ancillary-service price download and processing

This notebook downloads the full-year price inputs used by the MPC for the `AS_CAISO_EXP` region: day-ahead market (DAM) and real-time market (RTM) ancillary-service clearing prices. It uses CAISO OASIS directly, without requiring the legacy `caiso.py`/Tabula dependency.

Conventions:
- OASIS AS prices remain in $/MW in the saved clean files. The MPC converts them to its $/kWh settlement coefficients according to award duration.
- Six products are retained: non-spinning reserve, regulation down/up capacity, regulation down/up mileage, and spinning reserve.
- Correct Jun-Nov legacy data are reused; missing dates are downloaded.
- Clean files use a fixed California local clock: 24 hourly DAM rows and 96 15-minute RTM rows per day. Spring gaps are interpolated and repeated fall intervals are averaged.

Author: Yizhan Gu (UCSD CER)


In [ ]:
# Configuration and imports
import io
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

YEAR = 2025
START_DATE = pd.Timestamp(f"{YEAR}-01-01")
END_DATE = pd.Timestamp(f"{YEAR}-12-31")
LOCAL_TIMEZONE = "America/Los_Angeles"
AS_REGION = "AS_CAISO_EXP"
OASIS_URL = "https://oasis.caiso.com/oasisapi/SingleZip"
FORCE_REDOWNLOAD = False
REQUEST_PAUSE_SECONDS = 0.25
MAX_RETRIES = 5

PROJECT_ROOT = Path("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024")
PRODUCT_CODES = {
    "NR": "Non-Spinning Reserves",
    "RD": "Regulation Down",
    "RMD": "Regulation Mileage Down",
    "RMU": "Regulation Mileage Up",
    "RU": "Regulation Up",
    "SR": "Spinning Reserves",
}
CLEAN_NAMES = {
    "Non-Spinning Reserves": "NonSpin",
    "Regulation Down": "RegDown",
    "Regulation Mileage Down": "RegDownMileage",
    "Regulation Mileage Up": "RegUpMileage",
    "Regulation Up": "RegUp",
    "Spinning Reserves": "Spin",
}
MARKETS = {
    "DAM": {"queryname": "PRC_AS", "version": 12, "frequency": "1h", "rows_per_day": 24},
    "RTM": {"queryname": "PRC_INTVL_AS", "version": 1, "frequency": "15min", "rows_per_day": 96},
}

for market in MARKETS:
    output_dir = PROJECT_ROOT / "2025Data" / f"AS_{market}"
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "raw_oasis").mkdir(parents=True, exist_ok=True)

print(f"Target: {START_DATE.date()} through {END_DATE.date()}, region {AS_REGION}")


In [ ]:
# Download, legacy-reuse, and daylight-saving helpers
def local_day_utc_bounds(day):
    day = pd.Timestamp(day).normalize()
    start_local = day.tz_localize(LOCAL_TIMEZONE)
    end_local = (day + pd.DateOffset(days=1)).tz_localize(LOCAL_TIMEZONE)
    return start_local.tz_convert("UTC"), end_local.tz_convert("UTC")


def split_oasis_window(start_utc, end_utc):
    pieces = []
    cursor = start_utc
    while cursor < end_utc:
        next_cursor = min(cursor + pd.Timedelta(hours=24), end_utc)
        pieces.append((cursor, next_cursor))
        cursor = next_cursor
    return pieces


def oasis_time(ts):
    return pd.Timestamp(ts).tz_convert("UTC").strftime("%Y%m%dT%H:%M-0000")


def request_oasis_csv(params):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(OASIS_URL, params=params, timeout=90)
            response.raise_for_status()
            payload = io.BytesIO(response.content)
            if not zipfile.is_zipfile(payload):
                message = response.content[:800].decode("utf-8", errors="replace")
                raise RuntimeError(f"OASIS did not return a ZIP file: {message}")
            payload.seek(0)
            with zipfile.ZipFile(payload) as archive:
                csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
                if not csv_names:
                    raise RuntimeError(f"No CSV found in OASIS response: {archive.namelist()}")
                frames = [pd.read_csv(archive.open(name)) for name in csv_names]
            return pd.concat(frames, ignore_index=True)
        except Exception as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(min(2 ** (attempt - 1), 12))
    raise RuntimeError(f"OASIS request failed after {MAX_RETRIES} attempts") from last_error


def expected_physical_rows(day, frequency):
    start_utc, end_utc = local_day_utc_bounds(day)
    minutes = int((end_utc - start_utc) / pd.Timedelta(minutes=1))
    step_minutes = int(pd.Timedelta(frequency) / pd.Timedelta(minutes=1))
    return minutes // step_minutes


def long_oasis_to_wide(raw, market):
    raw = raw[(raw["ANC_REGION"] == AS_REGION) & (raw["MARKET_RUN_ID"] == market)].copy()
    pivot = raw.pivot_table(index="INTERVALSTARTTIME_GMT", columns="ANC_TYPE", values="MW", aggfunc="mean")
    missing = sorted(set(PRODUCT_CODES) - set(pivot.columns))
    if missing:
        raise ValueError(f"{market}: missing AS product codes {missing}")
    pivot = pivot[list(PRODUCT_CODES)].rename(columns=PRODUCT_CODES).reset_index().rename(columns={"INTERVALSTARTTIME_GMT": "Time"})
    pivot["Region"] = AS_REGION
    pivot["Market"] = market
    return pivot


def load_legacy_raw(market):
    path = PROJECT_ROOT / "2025Data" / f"AS_{market}" / f"AS_price_{YEAR}.csv"
    if not path.exists():
        return None
    frame = pd.read_csv(path, low_memory=False)
    required = {"Time", "Region", "Market", *PRODUCT_CODES.values()}
    if not required.issubset(frame.columns):
        return None
    frame = frame[(frame["Region"] == AS_REGION) & (frame["Market"] == market)].copy()
    frame["_utc"] = pd.to_datetime(frame["Time"], utc=True, errors="coerce")
    frame = frame.dropna(subset=["_utc"])
    frame["_local_date"] = frame["_utc"].dt.tz_convert(LOCAL_TIMEZONE).dt.date
    return frame


LEGACY_RAW = {market: load_legacy_raw(market) for market in MARKETS}


def load_saved_oasis_day(day, market):
    path = PROJECT_ROOT / "2025Data" / f"AS_{market}" / "raw_oasis" / f"{pd.Timestamp(day):%Y%m%d}.csv"
    if FORCE_REDOWNLOAD or not path.exists():
        return None
    try:
        raw = pd.read_csv(path)
        return long_oasis_to_wide(raw, market)
    except Exception:
        return None


def load_existing_day(day, market):
    saved = load_saved_oasis_day(day, market)
    if saved is not None:
        return saved, "saved_oasis"
    legacy = LEGACY_RAW[market]
    if FORCE_REDOWNLOAD or legacy is None:
        return None, None
    day_rows = legacy[legacy["_local_date"] == pd.Timestamp(day).date()].copy()
    expected = expected_physical_rows(day, MARKETS[market]["frequency"])
    if day_rows["_utc"].nunique() != expected:
        return None, None
    day_rows["Time"] = day_rows["_utc"].astype(str)
    return day_rows[["Time", "Region", "Market", *PRODUCT_CODES.values()]], "legacy"


def download_as_day(day, market):
    cfg = MARKETS[market]
    start_utc, end_utc = local_day_utc_bounds(day)
    chunks = []
    for chunk_start, chunk_end in split_oasis_window(start_utc, end_utc):
        params = {
            "resultformat": 6, "queryname": cfg["queryname"], "version": cfg["version"],
            "startdatetime": oasis_time(chunk_start), "enddatetime": oasis_time(chunk_end),
            "market_run_id": market, "anc_type": "ALL", "anc_region": AS_REGION,
        }
        chunks.append(request_oasis_csv(params))
        time.sleep(REQUEST_PAUSE_SECONDS)
    raw = pd.concat(chunks, ignore_index=True).drop_duplicates().reset_index(drop=True)
    path = PROJECT_ROOT / "2025Data" / f"AS_{market}" / "raw_oasis" / f"{pd.Timestamp(day):%Y%m%d}.csv"
    raw.to_csv(path, index=False)
    wide = long_oasis_to_wide(raw, market)
    expected = expected_physical_rows(day, cfg["frequency"])
    if pd.to_datetime(wide["Time"], utc=True).nunique() != expected:
        raise ValueError(f"{day:%Y-%m-%d} {market}: expected {expected} physical intervals, got {len(wide)}")
    return wide


def normalize_as_day(frame, day, market):
    cfg = MARKETS[market]
    frame = frame.copy()
    utc = pd.to_datetime(frame["Time"], utc=True, errors="raise")
    frame["_local"] = utc.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)
    product_columns = list(PRODUCT_CODES.values())
    for column in product_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    physical_rows = int(utc.nunique())
    local = frame.groupby("_local")[product_columns].mean().sort_index()
    target = pd.date_range(pd.Timestamp(day).normalize(), periods=cfg["rows_per_day"], freq=cfg["frequency"])
    local = local.reindex(target).interpolate(method="time", limit_direction="both")
    if local.isna().any().any():
        raise ValueError(f"{day:%Y-%m-%d} {market}: NaN remains after DST normalization")
    if physical_rows < cfg["rows_per_day"]:
        dst_action = "spring_gap_interpolated"
    elif physical_rows > cfg["rows_per_day"]:
        dst_action = "fall_duplicate_averaged"
    else:
        dst_action = "none"
    clean = local.rename(columns=CLEAN_NAMES).reset_index().rename(columns={"index": "datetime"})
    return clean, physical_rows, dst_action


In [ ]:
# Run the full-year missing-only download and create canonical raw and clean files
dates = pd.date_range(START_DATE, END_DATE, freq="D")
qc_rows = []
clean_days = {market: [] for market in MARKETS}
physical_days = {market: [] for market in MARKETS}

for day in tqdm(dates, desc="AS operating days"):
    for market in MARKETS:
        frame, source = load_existing_day(day, market)
        if frame is None:
            frame = download_as_day(day, market)
            source = "oasis_download"
        clean, physical_rows, dst_action = normalize_as_day(frame, day, market)
        clean_days[market].append(clean)
        physical_days[market].append(frame[["Time", "Region", "Market", *PRODUCT_CODES.values()]])
        qc_rows.append({
            "date": day.strftime("%Y-%m-%d"), "market": market, "source": source,
            "physical_rows": physical_rows, "model_rows": len(clean),
            "dst_action": dst_action, "missing_values": int(clean.isna().sum().sum()), "purpose": "study_year",
        })

# A 24-hour Rolling MPC on Dec 31 also needs Jan 1 of the following year.
boundary_day = END_DATE + pd.DateOffset(days=1)
for market in MARKETS:
    frame, source = load_existing_day(boundary_day, market)
    if frame is None:
        frame = download_as_day(boundary_day, market)
        source = "oasis_download"
    boundary_clean, physical_rows, dst_action = normalize_as_day(frame, boundary_day, market)
    boundary_path = PROJECT_ROOT / "2025Data" / f"AS_{market}" / f"AS_price_{boundary_day.year}_boundary.csv"
    boundary_clean.to_csv(boundary_path, index=False)
    qc_rows.append({
        "date": boundary_day.strftime("%Y-%m-%d"), "market": market, "source": source,
        "physical_rows": physical_rows, "model_rows": len(boundary_clean),
        "dst_action": dst_action, "missing_values": int(boundary_clean.isna().sum().sum()), "purpose": "rolling_boundary",
    })

AS_QC = pd.DataFrame(qc_rows)
for market in MARKETS:
    output_dir = PROJECT_ROOT / "2025Data" / f"AS_{market}"
    clean = pd.concat(clean_days[market], ignore_index=True)
    physical = pd.concat(physical_days[market], ignore_index=True).drop_duplicates(subset=["Time"], keep="last")
    clean.to_csv(output_dir / f"AS_price_{YEAR}_clear.csv", index=False)
    physical.to_csv(output_dir / f"AS_price_{YEAR}_{AS_REGION}.csv", index=False)
    market_qc = AS_QC[AS_QC["market"] == market]
    market_qc.to_csv(output_dir / f"AS_price_{YEAR}_download_qc.csv", index=False)
    expected = len(dates) * MARKETS[market]["rows_per_day"]
    timestamp = pd.to_datetime(clean["datetime"], errors="raise")
    assert len(clean) == expected
    assert timestamp.nunique() == expected
    assert clean.drop(columns="datetime").notna().all().all()
    print(f"{market}: {len(clean):,} model-clock prices, {timestamp.min()} to {timestamp.max()}")

print("Download sources:")
display(AS_QC.groupby(["market", "source"]).size().rename("days").reset_index())
print("DST normalization:")
display(AS_QC[AS_QC["dst_action"] != "none"])


In [ ]:
# Summary statistics and 300-dpi diagnostic figures
colors = {"NonSpin": "#3B82A0", "RegDown": "#D97757", "RegUp": "#4F9D69", "Spin": "#8B6FA8"}
for market in MARKETS:
    output_dir = PROJECT_ROOT / "2025Data" / f"AS_{market}"
    frame = pd.read_csv(output_dir / f"AS_price_{YEAR}_clear.csv")
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    stats = frame.drop(columns="datetime").describe()
    stats.to_csv(output_dir / f"AS_price_{YEAR}_stats.csv")
    print(f"{market} ($/MW):")
    display(stats)
    fig, ax = plt.subplots(figsize=(14, 5))
    for column, color in colors.items():
        ax.plot(frame["datetime"], frame[column], lw=0.55, alpha=0.8, label=column, color=color)
    ax.set_title(f"CAISO {market} ancillary-service prices ({YEAR})", fontweight="bold")
    ax.set_ylabel("$/MW")
    ax.set_xlabel(str(YEAR))
    ax.grid(alpha=0.25)
    ax.legend(ncol=4, frameon=False)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    fig.tight_layout()
    fig.savefig(output_dir / f"AS_price_{YEAR}_plot.png", dpi=300, bbox_inches="tight")
    plt.show()
